# Train a Logistic Regression Model (with Encoders Saved Separately)

This notebook shows the **training side** of deploying an ML model with Streamlit.

It does four things:
1. **Creates sample data** (so you can run this without downloading anything).
2. **Includes non-numeric (text) columns** alongside numeric ones.
3. **Encodes each text column and saves the encoder separately** to its own `.pkl` file.
4. **Trains a Logistic Regression model** and saves it.

At the end you will have these files, ready to upload to GitHub for your Streamlit app:

- `model.pkl` &rarr; the trained model
- `city_encoder.pkl` &rarr; encoder for the *City* column
- `membership_encoder.pkl` &rarr; encoder for the *Membership* column

> **Why save encoders separately?** Your model only understands numbers. During training,
> words like `"Paris"` are turned into numbers. The app must reuse the *exact same* mapping,
> so we save each encoder and reload it later &mdash; never re-typing the numbers by hand.


## Step 0 — Imports

We use `pandas` for the table of data, `numpy` for random numbers, `scikit-learn`
for the encoder and model, and `pickle` to save everything to files.


In [25]:
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Make the random data reproducible (same numbers every run)
np.random.seed(42)

## Step 1 — Create sample data

We build a small, made-up dataset of customers. It has:

- **Numeric columns:** `age`, `income`
- **Text (non-numeric) columns:** `city`, `membership`
- **Target we want to predict:** `purchased` (1 = bought, 0 = did not)

In a real project you would load your own CSV here instead
(e.g. `df = pd.read_csv("my_data.csv")`).


In [26]:
n = 500  # number of rows

df = pd.DataFrame({
    "age": np.random.randint(18, 70, size=n),
    "income": np.random.randint(20000, 120000, size=n),
    "city": np.random.choice(["London", "Paris", "Tokyo", "Delhi"], size=n),
    "membership": np.random.choice(["Basic", "Premium"], size=n),
})

# Create a target that loosely depends on the features, so the model has a pattern to learn.
# (Premium members, higher income, and younger customers are a bit more likely to purchase.)
score = (
    (df["income"] / 120000)
    + (df["membership"] == "Premium") * 0.4
    + (df["age"] < 40) * 0.2
    + np.random.normal(0, 0.3, size=n)
)
df["purchased"] = (score > score.mean()).astype(int)

df.head()

,age,income,city,membership,purchased
0,56,23343,London,Basic,0
1,69,33500,London,Premium,0
2,46,73222,Delhi,Basic,1
3,32,49375,London,Basic,0
4,60,29662,Delhi,Premium,0


Let's quickly look at the column types and check the class balance of the target.

In [27]:
print("Column data types:")
print(df.dtypes)
print("\nHow many purchased vs not:")
print(df["purchased"].value_counts())

Column data types:
age            int32
income         int32
city          object
membership    object
purchased      int64
dtype: object

How many purchased vs not:
purchased
0    260
1    240
Name: count, dtype: int64


## Step 2 — Encode the text columns and save each encoder

`LogisticRegression` cannot read words, so we convert each text column into numbers
using a `LabelEncoder`. We fit **one encoder per text column** and immediately
**save it to its own file**.

`fit_transform` does two jobs at once: it *learns* the word&rarr;number mapping and
*applies* it to the column.


In [28]:
# --- City column ---
city_encoder = LabelEncoder()
df["city_encoded"] = city_encoder.fit_transform(df["city"])

# --- Membership column ---
membership_encoder = LabelEncoder()
df["membership_encoded"] = membership_encoder.fit_transform(df["membership"])

# Peek at what each encoder learned. The position in this list IS the number assigned.
print("City mapping:")
for i, label in enumerate(city_encoder.classes_):
    print(f"  {label} -> {i}")

print("\nMembership mapping:")
for i, label in enumerate(membership_encoder.classes_):
    print(f"  {label} -> {i}")

df[["city", "city_encoded", "membership", "membership_encoded"]].head()

City mapping:
  Delhi -> 0
  London -> 1
  Paris -> 2
  Tokyo -> 3

Membership mapping:
  Basic -> 0
  Premium -> 1


,city,city_encoded,membership,membership_encoded
0,London,1,Basic,0
1,London,1,Premium,1
2,Delhi,0,Basic,0
3,London,1,Basic,0
4,Delhi,0,Premium,1


Now save each encoder to its own `.pkl` file. Upload these to GitHub next to `app.py`
so your Streamlit app can reload the identical mapping.


In [29]:
with open("city_encoder.pkl", "wb") as f:
    pickle.dump(city_encoder, f)

with open("membership_encoder.pkl", "wb") as f:
    pickle.dump(membership_encoder, f)

print("Saved: city_encoder.pkl, membership_encoder.pkl")

Saved: city_encoder.pkl, membership_encoder.pkl


## Step 3 — Prepare features (X) and target (y)

We feed the model only numeric columns: the two original numeric ones plus the two
**encoded** versions of the text columns. We drop the original word columns and the target.

> **Column order matters.** Remember this exact order &mdash; `age, income, city_encoded,
> membership_encoded`. Your Streamlit app must send the inputs in the *same* order.


In [30]:
feature_columns = ["age", "income", "city_encoded", "membership_encoded"]

X = df[feature_columns]
y = df["purchased"]

X.head()

,age,income,city_encoded,membership_encoded
0,56,23343,1,0
1,69,33500,1,1
2,46,73222,0,0
3,32,49375,1,0
4,60,29662,0,1


## Step 4 — Split into training and test sets

We hold back 20% of the data as a test set so we can honestly measure how well the
model does on rows it has never seen.


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows: ", X_test.shape[0])

Training rows: 400
Testing rows:  100


## Step 5 — Train the Logistic Regression model

`max_iter` is raised a little to make sure the model has enough iterations to settle.


In [32]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model trained.")

Model trained.


## Step 6 — Check how well it did

Accuracy is the share of correct predictions. The classification report adds precision
and recall for each class. Because this is random sample data, don't expect perfect
scores &mdash; the point is that the whole pipeline works end to end.


In [33]:
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("\nDetailed report:")
print(classification_report(y_test, predictions))

Accuracy: 0.8

Detailed report:
              precision    recall  f1-score   support

           0       0.85      0.82      0.83        61
           1       0.73      0.77      0.75        39

    accuracy                           0.80       100
   macro avg       0.79      0.79      0.79       100
weighted avg       0.80      0.80      0.80       100



## Step 7 — Save the trained model

Save the model to `model.pkl`. After this cell you will have all three files needed
for deployment.


In [34]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Saved: model.pkl")

Saved: model.pkl


In [35]:
# Get current working directory
cwd = os.getcwd()
print(cwd)

C:\Users\rkshw
